In [23]:
import pandas as pd
import librosa
import random
import numpy as np
from transformers import ASTForAudioClassification
from sklearn.model_selection import train_test_split

In [3]:
genres = [f"genre_{i}" for i in range(10)]

recipes = []

for genre in genres:
    for i in range(100):
        recipe = {
            "genre": genre,
            "file_path": f"/data/{genre}/audio_{i}.wav"
        }
        recipes.append(recipe)

In [4]:
train_recipes, val_recipes = train_test_split(
    recipes,
    test_size=0.2,
    shuffle=True,
    random_state=42
)

In [5]:
# Question 1

print("Total samples:", len(recipes))
print("Train samples:", len(train_recipes))
print("Validation samples:", len(val_recipes))

Total samples: 1000
Train samples: 800
Validation samples: 200


In [16]:
import os

AUDIO_DIR = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio"

all_files = [os.path.join(AUDIO_DIR, f) for f in os.listdir(AUDIO_DIR) if f.endswith(".wav")]

print("Total audio files:", len(all_files))

Total audio files: 2000


In [17]:

def sample_audio_paths(all_files):
    selected = random.sample(all_files, 5)  
    
    stem_paths = selected[:4]
    noise_path = selected[4]
    
    return stem_paths, noise_path

In [18]:
# Question 2

stem_paths, noise_path = sample_audio_paths(all_files)

final_audio = generate_final_audio(stem_paths, noise_path)

print(final_audio.shape)

(160000,)


In [19]:
from transformers import AutoFeatureExtractor
import numpy as np

extractor = AutoFeatureExtractor.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593"
)

mix = np.ones(160000, dtype=np.float32)

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

In [22]:
# Question 3

inputs = extractor(
    mix,
    sampling_rate=16000,
    return_tensors="pt"
)

input_values = inputs["input_values"].squeeze(0)
print(input_values.shape)

torch.Size([1024, 128])


In [24]:
# Question 4

model = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    num_labels=10,
    ignore_mismatched_sizes=True
)


num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(num_params)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                        
------------------------+----------+----------------------------------------------------------------------------------------
classifier.dense.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527, 768]) vs model:torch.Size([10, 768])
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527]) vs model:torch.Size([10])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


86196490


In [25]:
# Question 5

y_test = np.array([-0.85, 0.40, 0.20, -0.10])

y_normalized = y_test / (np.max(np.abs(y_test)) + 1e-9)

result = round(y_normalized[0], 3)

print("Normalized array:", y_normalized)
print("Value at index 0 (rounded):", result)

Normalized array: [-1.          0.47058823  0.23529412 -0.11764706]
Value at index 0 (rounded): -1.0
